# Multiple customer classes in a model

In this example, the model has three customer classes. The customer class determines the arrival distribution and service distribution.

The JSON for this built-in example can be loaded using `json2ciw.datasets.load_three_classes_model`.

## Imports

In [1]:
import json

import ciw
from rich import print

from json2ciw.datasets import load_paeds_pathway_model
from json2ciw.engine import CiwConverter, multiple_replications
from json2ciw.results import summarise_results, summarise_results_by_class, tidy_to_wide_format, tidy_to_wide_format_by_class
from json2ciw.schema import ProcessModel

## Load JSON

In [2]:
json_network = load_paeds_pathway_model()
print(json.dumps(json_network, indent=2))

{
  "name": "Paediatric clinic",
  "description": "A 24-hour paediatric clinic with babies and children sharing reception before routing to separate
specialist clinics. Babies register for an average of 15 minutes and children for an average of 10 minutes. Both 
specialist appointment types last an average of one hour.",
  "customer_classes": [
    {
      "name": "Baby",
      "label": "Babies"
    },
    {
      "name": "Child",
      "label": "Children"
    }
  ],
  "activities": [
    {
      "name": "Reception",
      "type": "activity",
      "resource": {
        "name": "Receptionist",
        "capacity": 1
      },
      "arrival_distribution": {
        "by_class": {
          "Baby": {
            "type": "exponential",
            "parameters": {
              "rate": 1.0
            }
          },
          "Child": {
            "type": "exponential",
            "parameters": {
              "rate": 2.0
            }
          }
        }
      },
      "service_distribution": {
        "by_class": {
          "Baby": {
            "type": "exponential",
            "parameters": {
              "rate": 4.0
            }
          },
          "Child": {
            "type": "exponential",
            "parameters": {
              "rate": 6.0
            }
          }
        }
      }
    },
    {
      "name": "Baby Specialist Clinic",
      "type": "activity",
      "resource": {
        "name": "Baby specialists",
        "capacity": 2
      },
      "service_distribution": {
        "by_class": {
          "Baby": {
            "type": "exponential",
            "parameters": {
              "rate": 1.0
            }
          }
        }
      }
    },
    {
      "name": "Children's Specialist Clinic",
      "type": "activity",
      "resource": {
        "name": "Children's specialists",
        "capacity": 3
      },
      "service_distribution": {
        "by_class": {
          "Child": {
            "type": "exponential",
            "parameters": {
              "rate": 1.0
            }
          }
        }
      }
    }
  ],
  "transitions": [
    {
      "from": "Reception",
      "to": "Baby Specialist Clinic",
      "probability": {
        "by_class": {
          "Baby": 1.0,
          "Child": 0.0
        }
      }
    },
    {
      "from": "Reception",
      "to": "Children's Specialist Clinic",
      "probability": {
        "by_class": {
          "Baby": 0.0,
          "Child": 1.0
        }
      }
    },
    {
      "from": "Baby Specialist Clinic",
      "to": "Exit",
      "probability": 1.0
    },
    {
      "from": "Children's Specialist Clinic",
      "to": "Exit",
      "probability": 1.0
    }
  ]
}

## Validate with `ProcessModel`

In [3]:
model_instance = ProcessModel(**json_network)

In [4]:
print(model_instance)

ProcessModel(
    name='Paediatric clinic',
    description='A 24-hour paediatric clinic with babies and children sharing reception before routing to separate 
specialist clinics. Babies register for an average of 15 minutes and children for an average of 10 minutes. Both 
specialist appointment types last an average of one hour.',
    customer_classes=[CustomerClass(name='Baby', label='Babies'), CustomerClass(name='Child', label='Children')],
    activities=[
        Activity(
            name='Reception',
            type='activity',
            resource=Resource(name='Receptionist', capacity=1),
            service_distribution=ClassDistributionMap(
                by_class={
                    'Baby': Distribution(type='exponential', parameters={'rate': 4.0}),
                    'Child': Distribution(type='exponential', parameters={'rate': 6.0})
                }
            ),
            arrival_distribution=ClassDistributionMap(
                by_class={
                    'Baby': Distribution(type='exponential', parameters={'rate': 1.0}),
                    'Child': Distribution(type='exponential', parameters={'rate': 2.0})
                }
            ),
            renege_distribution=None
        ),
        Activity(
            name='Baby Specialist Clinic',
            type='activity',
            resource=Resource(name='Baby specialists', capacity=2),
            service_distribution=ClassDistributionMap(
                by_class={'Baby': Distribution(type='exponential', parameters={'rate': 1.0})}
            ),
            arrival_distribution=None,
            renege_distribution=None
        ),
        Activity(
            name="Children's Specialist Clinic",
            type='activity',
            resource=Resource(name="Children's specialists", capacity=3),
            service_distribution=ClassDistributionMap(
                by_class={'Child': Distribution(type='exponential', parameters={'rate': 1.0})}
            ),
            arrival_distribution=None,
            renege_distribution=None
        )
    ],
    transitions=[
        Transition(
            source='Reception',
            target='Baby Specialist Clinic',
            probability=ClassProbabilityMap(by_class={'Baby': 1.0, 'Child': 0.0})
        ),
        Transition(
            source='Reception',
            target="Children's Specialist Clinic",
            probability=ClassProbabilityMap(by_class={'Baby': 0.0, 'Child': 1.0})
        ),
        Transition(source='Baby Specialist Clinic', target='Exit', probability=1.0),
        Transition(source="Children's Specialist Clinic", target='Exit', probability=1.0)
    ]
)

In [5]:
model_instance.display_diagram(include_resources=False, show_class_arrivals=True)

```mermaid 
graph TD
    Arrivals_Reception_Baby("Babies</br>Time between arrivals<br/>Exponential(λ=1.0)")
    Arrivals_Reception_Child("Children</br>Time between arrivals<br/>Exponential(λ=2.0)")
    Reception["Reception</br>Class-specific service distributions (n=2)"]
    Baby_Specialist_Clinic["Baby Specialist Clinic</br>Class-specific service distributions (n=1)"]
    Children's_Specialist_Clinic["Children's Specialist Clinic</br>Class-specific service distributions (n=1)"]
    Exit(["Exit"])

    Arrivals_Reception_Baby --> Reception
    Arrivals_Reception_Child --> Reception
 Reception -->|Babies: 100%<br/>Children: 0%| Baby_Specialist_Clinic
 Reception -->|Babies: 0%<br/>Children: 100%| Children's_Specialist_Clinic
 Baby_Specialist_Clinic --> Exit
 Children's_Specialist_Clinic --> Exit 
```

In [6]:
model_instance.get_distributions_df()

,Activity,Phase,Customer Class,Customer Class Label,Distribution Type,Parameters
0,Reception,Arrival,Baby,Babies,Exponential,rate=1.0
1,Reception,Arrival,Child,Children,Exponential,rate=2.0
2,Reception,Service,Baby,Babies,Exponential,rate=4.0
3,Reception,Service,Child,Children,Exponential,rate=6.0
4,Baby Specialist Clinic,Service,Baby,Babies,Exponential,rate=1.0
5,Children's Specialist Clinic,Service,Child,Children,Exponential,rate=1.0


In [7]:
model_instance.get_routing_matrix_df()

Reception  \
Customer Class Source Activity                           
Baby           Reception                           0.0   
               Baby Specialist Clinic              0.0   
               Children's Specialist Clinic        0.0   
Child          Reception                           0.0   
               Baby Specialist Clinic              0.0   
               Children's Specialist Clinic        0.0   

                                             Baby Specialist Clinic  \
Customer Class Source Activity                                        
Baby           Reception                                        1.0   
               Baby Specialist Clinic                           0.0   
               Children's Specialist Clinic                     0.0   
Child          Reception                                        0.0   
               Baby Specialist Clinic                           0.0   
               Children's Specialist Clinic                     0.0   

                                             Children's Specialist Clinic  \
Customer Class Source Activity                                              
Baby           Reception                                              0.0   
               Baby Specialist Clinic                                 0.0   
               Children's Specialist Clinic                           0.0   
Child          Reception                                              1.0   
               Baby Specialist Clinic                                 0.0   
               Children's Specialist Clinic                           0.0   

                                             Exit  
Customer Class Source Activity                     
Baby           Reception                      0.0  
               Baby Specialist Clinic         1.0  
               Children's Specialist Clinic   1.0  
Child          Reception                      0.0  
               Baby Specialist Clinic         1.0  
               Children's Specialist Clinic   1.0

In [8]:
model_instance.get_resources_df()

,Resource,Activity,Count
0,Receptionist,Reception,1
1,Baby specialists,Baby Specialist Clinic,2
2,Children's specialists,Children's Specialist Clinic,3


## Convert to `ciw` parameters

In [9]:
adapter = CiwConverter(model_instance)
network_params = adapter.generate_params()
print(network_params)

{
    'number_of_servers': [1, 2, 3],
    'arrival_distributions': {
        'Baby': [Exponential(rate=1.0), None, None],
        'Child': [Exponential(rate=2.0), None, None]
    },
    'service_distributions': {
        'Baby': [Exponential(rate=4.0), Exponential(rate=1.0), Deterministic(value=0.0)],
        'Child': [Exponential(rate=6.0), Deterministic(value=0.0), Exponential(rate=1.0)]
    },
    'routing': {
        'Baby': [[0.0, 1.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
        'Child': [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    }
}

## Build and run the `ciw` model

In [10]:
network = ciw.create_network(**network_params)
sim = ciw.Simulation(network)
sim.simulate_until_max_time(50)
print("Quick simulation run worked!")

Quick simulation run worked!

## Run the model for multiple replications

In [ ]:
df_reps = multiple_replications(
    network,
    model_instance,
    num_reps=5,
    runtime=2880,
    warmup=1440,
    n_jobs=-1,
)

df_reps.head()

## Convert to wide format

In [ ]:
# overall results
wide = tidy_to_wide_format(df_reps)
wide.head()

In [ ]:
# results by class = server strokes
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="severe_stroke")
wide_by_class.head()

In [ ]:
# results by class = tias
wide_by_class = tidy_to_wide_format_by_class(df_reps, customer_class="tia")
wide_by_class.head()

## Summarise results

In [ ]:
df_reps.head(2)

In [ ]:
summary = summarise_results(df_reps)
summary.round(1)

In [ ]:
summary_class = summarise_results_by_class(df_reps)
summary_class.round(1)